# 01 — Download MAESTRO v3

Run this notebook **once** to download the MAESTRO dataset to Google Drive.
You never need to run it again — the dataset stays on Drive between sessions.

## What is MAESTRO?

**MAESTRO** (MIDI and Audio Edited for Synchronous TRacks and Organization) is
the standard benchmark dataset for automatic piano music transcription.

Key facts (Hawthorne et al. 2018b "MAESTRO Dataset"):
- **~200 hours** of professional piano performances
- **9 years** of International Piano-e-Competition recordings (2004–2018)
- Every piece has a **precisely time-aligned audio + MIDI pair** (sub-millisecond)
- **Standard splits**: train / validation / test — fixed for reproducible benchmarking
- Download size: **~16 GB** (audio + MIDI)

Paper: C. Hawthorne et al., "Enabling Factorized Piano Music Modeling and Generation
with the MAESTRO Dataset," ICLR 2019. https://arxiv.org/abs/1810.12247

## Set paths

In [ ]:
import os
import glob

DRIVE_ROOT   = "/content/drive/MyDrive/piano_amt"
MAESTRO_DIR  = f"{DRIVE_ROOT}/maestro-v3.0.0"
ZIP_PATH     = f"{DRIVE_ROOT}/maestro-v3.0.0.zip"

os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root : {DRIVE_ROOT}")
print(f"Dataset dir: {MAESTRO_DIR}")
print(f"ZIP path   : {ZIP_PATH}")

## Check if already downloaded

In [ ]:
csv_exists = bool(glob.glob(f"{MAESTRO_DIR}/*.csv"))
if os.path.exists(MAESTRO_DIR) and csv_exists:
    print("✓ MAESTRO already downloaded and extracted — skip to 'Verify dataset' below.")
else:
    print("Dataset not yet downloaded. Continue with the cells below.")

## Download (skip if already done)

Downloads from the official Google Magenta storage bucket.  
**Expected time: 10–30 minutes** depending on Colab network speed.  
The `-c` flag resumes interrupted downloads.

In [ ]:
if not os.path.exists(ZIP_PATH):
    print("Downloading MAESTRO v3.0.0 (~16 GB)...")
    !wget -c "https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0.zip" \
         -O "{ZIP_PATH}"
    print("\nDownload complete.")
else:
    print("ZIP already exists — skipping download.")

## Unzip

Extracts the ZIP to `DRIVE_ROOT/maestro-v3.0.0/`.  
**Expected time: 5–15 minutes**.

In [ ]:
if not os.path.exists(MAESTRO_DIR):
    print("Extracting ZIP...")
    !unzip -q "{ZIP_PATH}" -d "{DRIVE_ROOT}"
    print("Unzipped ✓")
else:
    print("Already unzipped ✓")

## Verify dataset

In [ ]:
import pandas as pd
import glob

csv_files = sorted(glob.glob(f"{MAESTRO_DIR}/*.csv"))
assert csv_files, f"No CSV found in {MAESTRO_DIR}"

df = pd.read_csv(csv_files[0])
print(f"CSV: {csv_files[0]}")
print(f"Columns: {list(df.columns)}")
print()

for split in ["train", "validation", "test"]:
    n = len(df[df["split"] == split])
    print(f"  {split:12s}: {n:4d} files")

total_hrs = df["duration"].sum() / 3600
print(f"\nTotal duration: {total_hrs:.1f} hours")
print(f"Total files   : {len(df)}")

## Optional: use a small subset for testing

When calling `MAESTRODataset` or `get_dataloader`, pass `max_files=N` to limit
the dataset to the first N files.  This is useful for quick testing before
building the full cache:

```python
ds = MAESTRODataset(MAESTRO_DIR, 'train', cache_dir, max_files=5)
```

The full dataset has 967 training files (~178 hours).  
Start with `max_files=3` to verify the pipeline, then build the full cache.